# Notebook 60 — Regime-Conditioned Latent ODE and Multi-Step Transport

Notebook 59 showed that a single global coefficient-space differential transport law is too crude.
Notebook 60 tests whether transport becomes simpler in low-dimensional latent coefficient space:

\[
z = P(\beta - \mu)
\]

\[
\frac{dz}{dk}=F_j(z)
\]

where `j` is a regime family such as `forcing_mode` or latent cluster.

We compare:

1. Static symbolic coefficient prediction
2. Global latent affine ODE
3. Forcing-conditioned latent affine ODE
4. Cluster-conditioned latent affine ODE
5. Affine drift-only baselines

Transport is evaluated by multi-step integration:

\[
k=3 \to 5 \to 7
\]

and backward:

\[
k=7 \to 5 \to 3
\]

## Main goals

1. Encode coefficient vectors into 2D/3D latent space.
2. Learn simple regime-conditioned latent flows.
3. Integrate latent trajectories across k.
4. Decode latent states back to coefficients.
5. Compare coefficient, field, and trajectory reconstruction errors.
6. Interpret learned transport operators.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed governing template and per-regime coefficients

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

## Latent coefficient manifold

Notebook 57 showed the coefficient manifold is nearly 3D. Here we test `latent_dim = 2` and `latent_dim = 3`.

In [ ]:
# PCA encoder / decoder helpers

def fit_latent_encoder(coef_table, latent_dim=3):
    scaler = StandardScaler()
    Y = coef_table[COEF_COLS].to_numpy(dtype=float)
    Ystd = scaler.fit_transform(Y)
    pca = PCA(n_components=latent_dim, random_state=42)
    Z = pca.fit_transform(Ystd)
    return scaler, pca, Z

def encode_beta(beta, scaler, pca):
    beta = np.asarray(beta, dtype=float).reshape(1, -1)
    return pca.transform(scaler.transform(beta))[0]

def decode_z(z, scaler, pca):
    z = np.asarray(z, dtype=float).reshape(1, -1)
    return scaler.inverse_transform(pca.inverse_transform(z))[0]

for dim in [2, 3]:
    scaler_tmp, pca_tmp, Z_tmp = fit_latent_encoder(coef_df, latent_dim=dim)
    print(f"latent_dim={dim}, explained variance:", pca_tmp.explained_variance_ratio_, "sum=", pca_tmp.explained_variance_ratio_.sum())

In [ ]:
# Default latent space for visualization: 2D

vis_scaler, vis_pca, Z2 = fit_latent_encoder(coef_df, latent_dim=2)
coef_df["z1"] = Z2[:, 0]
coef_df["z2"] = Z2[:, 1]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
coef_df["latent_cluster"] = kmeans.fit_predict(Z2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ["forcing_mode", "flow_mode", "latent_cluster"]):
    for val in sorted(coef_df[col].astype(str).unique()):
        sub = coef_df[coef_df[col].astype(str) == val]
        ax.scatter(sub["z1"], sub["z2"], label=val, s=40)
    ax.set_xlabel("z1")
    ax.set_ylabel("z2")
    ax.set_title(f"Latent coefficient manifold by {col}")
    ax.legend()

plt.tight_layout()
plt.show()

## Static symbolic baseline from Notebook 58

This remains the strongest predictor from the previous notebook, so all ODE models are compared against it.

In [ ]:
# Static symbolic feature construction

def build_static_symbolic_features(df_in, columns=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def fit_static_symbolic(train_coef_df, test_coef_df):
    X_train = build_static_symbolic_features(train_coef_df)
    X_test = build_static_symbolic_features(test_coef_df, columns=X_train.columns)

    Yhat = np.zeros((len(test_coef_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_coef_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(X_train)
        Xte_s = scaler.transform(X_test)

        model = LassoCV(cv=min(5, len(train_coef_df)), random_state=42, max_iter=20000)
        model.fit(Xtr_s, y_train)
        Yhat[:, j] = model.predict(Xte_s)

    return Yhat

## Build latent k-trajectories

Each trajectory is grouped by:

`system, task, forcing_mode, flow_mode`

and has states at k = 3, 5, 7 when available.

In [ ]:
# Build trajectory records

traj_group_cols = ["system", "task", "forcing_mode", "flow_mode"]

def build_latent_trajectories(coef_table, scaler, pca):
    trajectories = []
    for vals, sub in coef_table.groupby(traj_group_cols):
        sub = sub.sort_values("k").reset_index(drop=True)
        if sub["k"].nunique() < 2:
            continue

        betas = sub[COEF_COLS].to_numpy(dtype=float)
        zs = np.array([encode_beta(beta, scaler, pca) for beta in betas])
        ks = sub["k"].to_numpy(dtype=float)

        trajectories.append({
            "meta": vals,
            "system": vals[0],
            "task": vals[1],
            "forcing_mode": vals[2],
            "flow_mode": vals[3],
            "k": ks,
            "beta": betas,
            "z": zs,
            "rows": sub.copy(),
        })
    return trajectories

trajectories_2d = build_latent_trajectories(coef_df, vis_scaler, vis_pca)
print("Number of latent trajectories:", len(trajectories_2d))

In [ ]:
# Build finite-difference latent transport table

def build_latent_transport_df(trajectories, cluster_lookup=None):
    rows = []
    for tr in trajectories:
        ks = tr["k"]
        zs = tr["z"]
        betas = tr["beta"]
        rows_df = tr["rows"]

        for i in range(len(ks) - 1):
            k0, k1 = ks[i], ks[i + 1]
            dk = k1 - k0
            z0, z1 = zs[i], zs[i + 1]
            dzdk = (z1 - z0) / dk

            row = {
                "system": tr["system"],
                "task": tr["task"],
                "forcing_mode": tr["forcing_mode"],
                "flow_mode": tr["flow_mode"],
                "k0": k0,
                "k1": k1,
                "dk": dk,
                "regime_id_0": rows_df.loc[i, "regime_id"],
                "regime_id_1": rows_df.loc[i + 1, "regime_id"],
            }

            if cluster_lookup is not None:
                row["latent_cluster"] = cluster_lookup.get(rows_df.loc[i, "regime_id"], -1)

            for j, val in enumerate(z0):
                row[f"z{j+1}_0"] = val
            for j, val in enumerate(dzdk):
                row[f"dz{j+1}_dk"] = val
            for j, val in enumerate(betas[i]):
                row[f"{COEF_COLS[j]}_at_k0"] = val

            rows.append(row)

    return pd.DataFrame(rows)

cluster_lookup = dict(zip(coef_df["regime_id"], coef_df["latent_cluster"]))
transport_z_df = build_latent_transport_df(trajectories_2d, cluster_lookup=cluster_lookup)
display(transport_z_df.head())

## Latent ODE models

We fit affine latent flows:

\[
\frac{dz}{dk}=Az+b
\]

Conditioning options:
- global
- forcing mode
- latent cluster

Also test drift-only:

\[
\frac{dz}{dk}=b
\]

In [ ]:
# Latent ODE fit/predict helpers

def z_cols(latent_dim):
    return [f"z{i+1}_0" for i in range(latent_dim)]

def dz_cols(latent_dim):
    return [f"dz{i+1}_dk" for i in range(latent_dim)]

def fit_affine_flow(train_df, latent_dim, group_col=None, alpha=1.0, drift_only=False):
    models = {}
    default_key = "__global__"

    if group_col is None:
        groups = [(default_key, train_df)]
    else:
        groups = list(train_df.groupby(group_col))

    for key, sub in groups:
        if len(sub) < 2:
            continue

        if drift_only:
            # constant mean drift
            drift = sub[dz_cols(latent_dim)].to_numpy(dtype=float).mean(axis=0)
            models[key] = {"type": "drift", "drift": drift}
        else:
            X = sub[z_cols(latent_dim)].to_numpy(dtype=float)
            Y = sub[dz_cols(latent_dim)].to_numpy(dtype=float)
            model = Ridge(alpha=alpha)
            model.fit(X, Y)
            models[key] = {"type": "affine", "model": model}

    if len(models) == 0:
        # robust fallback
        drift = train_df[dz_cols(latent_dim)].to_numpy(dtype=float).mean(axis=0)
        models[default_key] = {"type": "drift", "drift": drift}

    return models

def predict_dz(models, z, key=None):
    if key in models:
        m = models[key]
    elif "__global__" in models:
        m = models["__global__"]
    else:
        # fallback to first model
        m = next(iter(models.values()))

    z = np.asarray(z, dtype=float).reshape(1, -1)

    if m["type"] == "drift":
        return np.asarray(m["drift"], dtype=float)
    return m["model"].predict(z)[0]

def integrate_latent_steps(z_start, k_start, k_targets, models, latent_dim, group_key=None):
    z = np.asarray(z_start, dtype=float).copy()
    k_curr = float(k_start)
    out = {k_curr: z.copy()}

    for k_next in k_targets:
        dk = float(k_next - k_curr)
        dzdk = predict_dz(models, z, key=group_key)
        z = z + dk * dzdk
        k_curr = float(k_next)
        out[k_curr] = z.copy()
    return out

## Multi-step evaluation

Evaluate forward and backward paths:

- forward: 3 → 5 → 7
- backward: 7 → 5 → 3

Models:
- static symbolic
- global latent affine ODE
- forcing-conditioned latent affine ODE
- cluster-conditioned latent affine ODE
- forcing drift-only ODE

In [ ]:
# Evaluation helpers

def regime_at_k(tr, kval):
    idxs = np.where(np.isclose(tr["k"], kval))[0]
    if len(idxs) == 0:
        return None
    i = int(idxs[0])
    return tr["rows"].loc[i, "regime_id"], tr["beta"][i], tr["z"][i]

def eval_beta_prediction(rid, beta_true, beta_hat):
    sub = regime_subsets[rid]
    y_emp = sub["predicted_flow"].to_numpy(dtype=float)
    pred = predict_with_beta(sub, beta_hat)
    return {
        "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
        "field_rmse": math.sqrt(mean_squared_error(y_emp, pred)),
        "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
    }

In [ ]:
def run_latent_ode_experiment(latent_dim=2):
    scaler, pca, Z = fit_latent_encoder(coef_df, latent_dim=latent_dim)
    temp_coef = coef_df.copy()
    for j in range(latent_dim):
        temp_coef[f"z{j+1}"] = Z[:, j]

    km = KMeans(n_clusters=3, random_state=42, n_init=20)
    temp_coef["latent_cluster"] = km.fit_predict(Z)
    cluster_lookup = dict(zip(temp_coef["regime_id"], temp_coef["latent_cluster"]))

    trajectories = build_latent_trajectories(temp_coef, scaler, pca)
    transport_df = build_latent_transport_df(trajectories, cluster_lookup=cluster_lookup)

    rows = []

    for holdout_direction in ["forward_3_to_7", "backward_7_to_3"]:
        for tr in trajectories:
            ks = list(tr["k"])

            if holdout_direction == "forward_3_to_7":
                if not all(k in ks for k in [3.0, 5.0, 7.0]):
                    continue
                k_start = 3.0
                k_targets = [5.0, 7.0]
            else:
                if not all(k in ks for k in [3.0, 5.0, 7.0]):
                    continue
                k_start = 7.0
                k_targets = [5.0, 3.0]

            start = regime_at_k(tr, k_start)
            if start is None:
                continue
            start_rid, beta_start, z_start = start

            target_rids = []
            target_betas = {}
            target_zs = {}
            for kt in k_targets:
                target = regime_at_k(tr, kt)
                if target is None:
                    continue
                rid, beta, z = target
                target_rids.append(rid)
                target_betas[kt] = beta
                target_zs[kt] = z

            # Train transport flows excluding this trajectory's finite-diff rows
            exclude_rids = set(tr["rows"]["regime_id"].tolist())
            train_transport = transport_df[
                ~transport_df["regime_id_0"].isin(exclude_rids)
                & ~transport_df["regime_id_1"].isin(exclude_rids)
            ].reset_index(drop=True)

            if len(train_transport) < 5:
                train_transport = transport_df.copy()

            global_models = fit_affine_flow(train_transport, latent_dim, group_col=None, alpha=1.0, drift_only=False)
            forcing_models = fit_affine_flow(train_transport, latent_dim, group_col="forcing_mode", alpha=1.0, drift_only=False)
            cluster_models = fit_affine_flow(train_transport, latent_dim, group_col="latent_cluster", alpha=1.0, drift_only=False)
            forcing_drift_models = fit_affine_flow(train_transport, latent_dim, group_col="forcing_mode", alpha=1.0, drift_only=True)

            model_specs = {
                "latent_global_affine": (global_models, "__global__"),
                "latent_forcing_affine": (forcing_models, tr["forcing_mode"]),
                "latent_cluster_affine": (cluster_models, cluster_lookup.get(start_rid, -1)),
                "latent_forcing_drift": (forcing_drift_models, tr["forcing_mode"]),
            }

            # Static symbolic baseline trained excluding target trajectory's regimes
            train_coef = temp_coef[~temp_coef["regime_id"].isin(exclude_rids)].reset_index(drop=True)
            if len(train_coef) < 5:
                train_coef = temp_coef.copy()

            for model_name, (models, key) in model_specs.items():
                z_path = integrate_latent_steps(z_start, k_start, k_targets, models, latent_dim, group_key=key)

                for kt in k_targets:
                    target = regime_at_k(tr, kt)
                    if target is None:
                        continue
                    rid, beta_true, z_true = target
                    beta_hat = decode_z(z_path[float(kt)], scaler, pca)
                    metrics = eval_beta_prediction(rid, beta_true, beta_hat)
                    metrics.update({
                        "latent_dim": latent_dim,
                        "direction": holdout_direction,
                        "model": model_name,
                        "source_regime": start_rid,
                        "target_regime": rid,
                        "target_k": kt,
                        "latent_rmse": math.sqrt(mean_squared_error(z_true, z_path[float(kt)])),
                    })
                    rows.append(metrics)

            # static symbolic direct predictions for same target points
            target_rows = []
            target_k_values = []
            for kt in k_targets:
                target = regime_at_k(tr, kt)
                if target is None:
                    continue
                rid, beta_true, z_true = target
                target_rows.append(temp_coef[temp_coef["regime_id"] == rid])
                target_k_values.append((kt, rid, beta_true, z_true))

            if len(target_rows) > 0:
                test_coef = pd.concat(target_rows, axis=0).reset_index(drop=True)
                beta_static_mat = fit_static_symbolic(train_coef, test_coef)
                for idx, (kt, rid, beta_true, z_true) in enumerate(target_k_values):
                    beta_hat = beta_static_mat[idx]
                    z_hat = encode_beta(beta_hat, scaler, pca)
                    metrics = eval_beta_prediction(rid, beta_true, beta_hat)
                    metrics.update({
                        "latent_dim": latent_dim,
                        "direction": holdout_direction,
                        "model": "static_symbolic",
                        "source_regime": start_rid,
                        "target_regime": rid,
                        "target_k": kt,
                        "latent_rmse": math.sqrt(mean_squared_error(z_true, z_hat)),
                    })
                    rows.append(metrics)

    return pd.DataFrame(rows), transport_df, temp_coef, scaler, pca

eval2_df, transport2_df, coef2_df, scaler2, pca2 = run_latent_ode_experiment(latent_dim=2)
eval3_df, transport3_df, coef3_df, scaler3, pca3 = run_latent_ode_experiment(latent_dim=3)

eval_df = pd.concat([eval2_df, eval3_df], ignore_index=True)
display(eval_df.head())

In [ ]:
summary_df = eval_df.groupby(["latent_dim", "direction", "model"])[
    ["coef_rmse", "field_rmse", "traj_rmse", "latent_rmse"]
].mean().reset_index()

display(summary_df.sort_values(["latent_dim", "direction", "traj_rmse"]))

In [ ]:
# Model comparison plots

for latent_dim in sorted(eval_df["latent_dim"].unique()):
    sub = summary_df[summary_df["latent_dim"] == latent_dim]
    for metric in ["coef_rmse", "field_rmse", "traj_rmse", "latent_rmse"]:
        pivot = sub.pivot_table(index="direction", columns="model", values=metric)
        ax = pivot.plot(kind="bar", figsize=(14, 5))
        ax.set_title(f"latent_dim={latent_dim} — {metric}")
        ax.set_ylabel(metric)
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()

## Forward/backward asymmetry

Does latent transport work better forward or backward?

In [ ]:
asym_df = eval_df.groupby(["latent_dim", "model", "direction"])[["traj_rmse", "field_rmse", "coef_rmse"]].mean().reset_index()
display(asym_df)

for metric in ["traj_rmse", "field_rmse", "coef_rmse"]:
    pivot = asym_df.pivot_table(index=["latent_dim", "model"], columns="direction", values=metric)
    if set(["forward_3_to_7", "backward_7_to_3"]).issubset(pivot.columns):
        pivot["asymmetry_forward_minus_backward"] = pivot["forward_3_to_7"] - pivot["backward_7_to_3"]
    display(pivot)

## Latent transport arrows

Visualize true finite-difference transport arrows in 2D latent space.

In [ ]:
plt.figure(figsize=(9, 7))

for fmode in sorted(coef2_df["forcing_mode"].astype(str).unique()):
    sub = coef2_df[coef2_df["forcing_mode"].astype(str) == fmode]
    plt.scatter(sub["z1"], sub["z2"], label=fmode, alpha=0.55)

for _, row in transport2_df.iterrows():
    z0 = np.array([row["z1_0"], row["z2_0"]], dtype=float)
    dz = np.array([row["dz1_dk"], row["dz2_dk"]], dtype=float) * row["dk"]
    plt.arrow(
        z0[0], z0[1],
        dz[0], dz[1],
        head_width=0.04,
        length_includes_head=True,
        alpha=0.35,
    )

plt.title("Latent coefficient transport arrows as k changes")
plt.xlabel("z1")
plt.ylabel("z2")
plt.legend()
plt.tight_layout()
plt.show()

## True vs integrated latent paths

Plot representative paths for one family under the best latent ODE model.

In [ ]:
# Identify best latent ODE model excluding static symbolic

latent_only = summary_df[summary_df["model"] != "static_symbolic"].copy()
best_row = latent_only.sort_values("traj_rmse").iloc[0]
best_dim = int(best_row["latent_dim"])
best_model_name = best_row["model"]

print("Best latent ODE model:", best_model_name, "latent_dim:", best_dim)

# Use 2D visualization even if best model is 3D: plot first two latent coords
if best_dim == 2:
    temp_coef = coef2_df
    transport_df = transport2_df
    scaler = scaler2
    pca = pca2
else:
    temp_coef = coef3_df
    transport_df = transport3_df
    scaler = scaler3
    pca = pca3

# Choose a trajectory with 3,5,7
trajs = build_latent_trajectories(temp_coef, scaler, pca)
example_tr = None
for tr in trajs:
    if all(k in list(tr["k"]) for k in [3.0, 5.0, 7.0]):
        example_tr = tr
        break

# Train appropriate model on all transport rows for visualization
if best_model_name == "latent_global_affine":
    models = fit_affine_flow(transport_df, best_dim, group_col=None, alpha=1.0, drift_only=False)
    key = "__global__"
elif best_model_name == "latent_forcing_affine":
    models = fit_affine_flow(transport_df, best_dim, group_col="forcing_mode", alpha=1.0, drift_only=False)
    key = example_tr["forcing_mode"]
elif best_model_name == "latent_cluster_affine":
    models = fit_affine_flow(transport_df, best_dim, group_col="latent_cluster", alpha=1.0, drift_only=False)
    rid0 = example_tr["rows"].iloc[0]["regime_id"]
    key = dict(zip(temp_coef["regime_id"], temp_coef["latent_cluster"])).get(rid0, -1)
else:
    models = fit_affine_flow(transport_df, best_dim, group_col="forcing_mode", alpha=1.0, drift_only=True)
    key = example_tr["forcing_mode"]

k_start = 3.0
k_targets = [5.0, 7.0]
start_idx = np.where(np.isclose(example_tr["k"], k_start))[0][0]
z_start = example_tr["z"][start_idx]
z_path = integrate_latent_steps(z_start, k_start, k_targets, models, best_dim, group_key=key)

true_z = {float(k): z for k, z in zip(example_tr["k"], example_tr["z"])}

plt.figure(figsize=(7, 6))
true_points = np.array([true_z[k] for k in sorted(true_z.keys())])
pred_points = np.array([z_path[k] for k in sorted(z_path.keys())])

plt.plot(true_points[:, 0], true_points[:, 1], marker="o", label="true latent path")
plt.plot(pred_points[:, 0], pred_points[:, 1], marker="x", linestyle="--", label="integrated latent path")

for k, z in true_z.items():
    plt.text(z[0], z[1], f"k={int(k)}", fontsize=9)

plt.title(f"True vs integrated latent path: {example_tr['forcing_mode']} | {example_tr['flow_mode']}")
plt.xlabel("z1")
plt.ylabel("z2")
plt.legend()
plt.tight_layout()
plt.show()

## Learned transport operators

For affine models:

\[
dz/dk = A z + b
\]

plot the learned matrices for each forcing mode.

In [ ]:
# Fit forcing-conditioned affine operators on all 2D transport data for interpretability

forcing_models_2d = fit_affine_flow(transport2_df, latent_dim=2, group_col="forcing_mode", alpha=1.0, drift_only=False)

for key, pack in forcing_models_2d.items():
    if pack["type"] != "affine":
        continue
    model = pack["model"]
    A = model.coef_
    b = model.intercept_

    fig, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(A, cmap="coolwarm")
    ax.set_title(f"A matrix: forcing={key}")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["z1", "z2"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["dz1/dk", "dz2/dk"])
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    print("forcing:", key)
    print("A =")
    print(A)
    print("b =", b)
    print()

## Final verdicts

In [ ]:
verdict_rows = []
for latent_dim in sorted(eval_df["latent_dim"].unique()):
    for direction in sorted(eval_df["direction"].unique()):
        sub = summary_df[(summary_df["latent_dim"] == latent_dim) & (summary_df["direction"] == direction)]
        best = sub.sort_values("traj_rmse").iloc[0]
        best_latent = sub[sub["model"] != "static_symbolic"].sort_values("traj_rmse").iloc[0]
        static = sub[sub["model"] == "static_symbolic"].iloc[0]

        verdict_rows.append({
            "latent_dim": latent_dim,
            "direction": direction,
            "best_overall": best["model"],
            "best_overall_traj_rmse": best["traj_rmse"],
            "best_latent_ode": best_latent["model"],
            "best_latent_traj_rmse": best_latent["traj_rmse"],
            "static_symbolic_traj_rmse": static["traj_rmse"],
            "latent_minus_static": best_latent["traj_rmse"] - static["traj_rmse"],
        })

verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)

## Working conclusion

Notebook 60 tests whether the coefficient dynamics become simple affine flows in latent space when conditioned by regime family.

A strong result is:
- forcing-conditioned latent ODE improves over global coefficient-space differential transport
- static symbolic transport remains the best coordinate-chart predictor
- latent ODE gives a readable dynamical description of coefficient motion
- learned `A` matrices provide interpretable transport operators per forcing family

If latent ODE wins or becomes close to static symbolic, Notebook 61 should analyze transport operators and reduce them symbolically.

If static symbolic remains much better, Notebook 61 should formalize the static symbolic law as the final coordinate chart.